# Phase 4 — Frozen encoder feature extraction

Extracts frozen **VideoMAE** (768-d) and **CLIP** (512-d) embeddings for every clip in train/val/test, plus aligns the shipped **SlowFast** (2304-d) features for the Phase 5 baseline. All three are idempotent — safe to re-run, already-cached clips are skipped.

**Note:** full extraction over ~3.2k clips is the slowest step in the pipeline. It's resumable, so it's fine to interrupt and continue later.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/ -> project/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from project.config import load_config, resolve_path

cfg = load_config()


In [2]:
from project.features.extract_slowfast_baseline import extract_slowfast_features

extract_slowfast_features(cfg)

  DS1/train: 0 SlowFast features saved


  DS1/val: 0 SlowFast features saved


  DS1/test: 0 SlowFast features saved


  DS2/train: 0 SlowFast features saved


  DS2/val: 0 SlowFast features saved


  DS3/train: 0 SlowFast features saved


  DS3/val: 0 SlowFast features saved
SlowFast features -> /Users/rafiabhista/Documents/code/apple/c1_audio/emomv_2/project/features/slowfast  (saved 0, already-cached 3178)


In [3]:
from project.models.encoders import get_device, FrozenVideoMAEEncoder, FrozenCLIPEncoder
from project.features.extract_features import extract_split

device = get_device(cfg['encoders']['device_preference'])
print('device:', device)

videomae_enc = FrozenVideoMAEEncoder(cfg['encoders']['videomae_model'], device=device)
clip_enc = FrozenCLIPEncoder(cfg['encoders']['clip_model'], device=device)

videomae_dir = resolve_path(cfg, 'paths.videomae_features_dir'); videomae_dir.mkdir(parents=True, exist_ok=True)
clip_dir = resolve_path(cfg, 'paths.clip_features_dir'); clip_dir.mkdir(parents=True, exist_ok=True)

for split_name, path_key in [('train','paths.train_csv'), ('val','paths.val_csv'), ('test','paths.test_csv')]:
    extract_split(resolve_path(cfg, path_key), split_name, cfg, videomae_enc, clip_enc, videomae_dir, clip_dir)

device: mps


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.norm.weight                                                  | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[train] 2232/2232 clips pending extraction


Extracting [train]: 100%|██████████| 279/279 [12:24<00:00,  2.67s/it]


[val] 479/479 clips pending extraction


Extracting [val]: 100%|██████████| 60/60 [02:54<00:00,  2.90s/it]


[test] 478/478 clips pending extraction


Extracting [test]: 100%|██████████| 60/60 [02:53<00:00,  2.89s/it]
